In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
import matplotlib.pyplot as plt
import numpy as np


train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2) # 128 is 2*7 so it's more hardware optimized
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

# neede denorm
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# enjoy 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))

imgs_indices = [0, 10, 20, 30, 67] #67

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]  # Load image  label
    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) -> (H, W, C) need for numpy
    # Denorm
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)
    # take this
    axes[i].imshow(img_np)
    axes[i].set_title(f'Label: {label}')
    axes[i].axis('off')
plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
import torch.nn as nn
from torchvision import models

# Load pretrained EfficientNetV2-S model
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
# Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model)

In [ ]:
# Write your code here
from tqdm import tqdm    # fun fact the one created the library meant تقدم

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)
        labels = labels - 1  # EMNIST letters labels are 1-26, shift to 0-25 it must starting from zero shout out frost ;)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        # 67 accuracy
        predictions = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(predictions, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            labels = labels - 1  # frost

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # 67 accuracy
            predictions = torch.softmax(outputs, dim=1)
            predictions = torch.argmax(predictions, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [ ]:
# Write your code here
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)
num_epochs = 20

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

In [ ]:
# Write your code here
def validate_tta(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader):
            images, labels = images.to(device), labels.to(device)
            labels = labels - 1  # frost
            # org pred
            outputs_original = model(images)
            # flip
            h_flipped = torch.flip(images, dims=[3])
            outputs_hflip = model(h_flipped)
            # flip vert
            v_flipped = torch.flip(images, dims=[2])
            outputs_vflip = model(v_flipped)
            # Average the 3 predictions
            outputs = (outputs_original + outputs_hflip + outputs_vflip) / 3.0
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            # 67 accuracy
            predictions = torch.softmax(outputs, dim=1)
            predictions = torch.argmax(predictions, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

tta_loss, tta_accuracy = validate_tta(model, test_loader, criterion, device)
print(f"TTA Val Loss={tta_loss:.4f}, TTA Val Accuracy={tta_accuracy:.2f}%")


val_loss, val_accuracy = validate(model, test_loader, criterion, device)
print(f"Standard Val Loss={val_loss:.4f}, Standard Val Accuracy={val_accuracy:.2f}%")